# 00 · Synthetic GPS dwells

Every demo runs on **synthetic** data: no real device ever enters this repository.
This notebook draws a small city (statistical areas, homes, offices, third places),
simulates three months of dwells for 300 devices and writes them in the same raw
schema as the real data (see `notebook_docs/data_schema.md`).

The generator is driven by the `synthetic` block of `configs/demo.json`:

* nights, Shabbat and Israeli holidays at home (calendar from `calendar` settings);
* office workers commute on workdays, except on remote days, whose probability is
  `remote_day_probability[month] × personal affinity`;
* on remote days some users work from a *work-friendly* third place (cafe, coworking,
  library) in the late morning; evening and weekend outings go to *leisure* places;
* every dwell is observed with a user-specific probability (data sparsity).

Because we know the truth, later notebooks can check how well each method recovers it.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from remote_work_utils.settings import load_settings, resolve_path
from remote_work_utils.synthetic import generate

settings = load_settings("../configs/demo.json")
synthetic = settings["synthetic"]
synthetic["months"], synthetic["n_users"]

## Generate and write the demo inputs
`pipelines/make_demo_data.py` does the same from the command line.

In [ ]:
world, months = generate(settings)

for key, frame in [("stat_areas", world.stat_areas), ("boundary", world.boundary)]:
    path = resolve_path(settings, key)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_file(path, driver="GeoJSON")
world.places.to_csv(resolve_path(settings, "places"), index=False)
world.users.to_csv(resolve_path(settings, "synthetic_users"), index=False)
for month, raw in months.items():
    path = resolve_path(settings, "dwells_raw", month=month)
    path.parent.mkdir(parents=True, exist_ok=True)
    raw.to_parquet(path, index=False)

pd.DataFrame(
    {month: {"dwells": len(raw), "devices": raw["identifier"].nunique()} for month, raw in months.items()}
).T

## The raw schema
One row per dwell, timestamps in UTC plus local time. `true_place_kind` / `true_place_id`
exist only in synthetic data and are never used by the methods.

In [ ]:
months["202004"].head()

## The synthetic city

In [ ]:
users = world.users
places = world.places

fig, ax = plt.subplots(figsize=(7, 8))
world.stat_areas.plot(ax=ax, column="population", cmap="Greys", alpha=0.5, edgecolor="white", legend=True,
                      legend_kwds={"label": "Population per statistical area", "shrink": 0.6})
ax.scatter(users["home_lon"], users["home_lat"], s=6, color="#2b6cb0", label="Home")
ax.scatter(users["office_lon"], users["office_lat"], s=10, marker="s", color="#c53030", label="Office")
for group, marker, color in [("work_friendly", "^", "#2f855a"), ("leisure", "v", "#dd6b20")]:
    subset = places[places["place_group"] == group]
    ax.scatter(subset["lon"], subset["lat"], s=40, marker=marker, color=color, edgecolor="black",
               label=f"Third place ({group.replace('_', '-')})")
ax.set_title("Synthetic city: homes, offices and third places")
ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Latitude (°)")
ax.legend(loc="lower left", fontsize=8)
plt.show()

> **Fig. 0.1.** Synthetic study area used by all demos. Homes are drawn in proportion to the
> population of each statistical area; offices cluster around a few business hubs.
> *Notes:* 300 synthetic devices, 40 third places; no real data.